# MK-UNet screening reproduction
AI-assisted code and experimental evidence for Yuvraj Verma. Review the protocol and interpret results in your own words before submission. Run cells sequentially on a CUDA GPU. Dataset contents are not redistributed.


In [ ]:
import sys, subprocess
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "A CUDA GPU is required"
print(torch.cuda.get_device_name())


## Install dependencies and preserve evidence
Use the environment described in README.md. Before reproducing, rename the provided runs/ directory to recorded-runs/ and start with a fresh runs/ directory. The supplied results must not be overwritten. The official architecture is pinned to a recorded Git commit.


In [ ]:
%pip install -r requirements-recorded.txt


In [ ]:
subprocess.run([sys.executable, 'download_data.py', 'clinic'], check=True)
subprocess.run([sys.executable, 'download_data.py', 'mass'], check=True)


In [ ]:
from pathlib import Path
assert not Path("runs/clinic_seed42").exists(), "Preserve the supplied run directory before reproducing."
subprocess.run([sys.executable, 'train_screening.py', '--dataset', 'clinic', '--epochs', '200', '--batch', '16', '--amp'], check=True)


In [ ]:
from pathlib import Path
assert not Path("runs/mass_seed42").exists(), "Preserve the supplied run directory before reproducing."
subprocess.run([sys.executable, 'train_screening.py', '--dataset', 'mass', '--epochs', '200', '--batch', '16'], check=True)


In [ ]:
subprocess.run([sys.executable, 'prototype_evaluation.py', 'clinic'], check=True)
subprocess.run([sys.executable, 'prototype_evaluation.py', 'mass'], check=True)


In [ ]:
import json
from pathlib import Path
for dataset in ["clinic", "mass"]:
    run = Path("runs") / f"{dataset}_seed42"
    result = json.loads((run / "results.json").read_text())
    print(dataset, "native raw Dice", result["native_raw"]["mean_dice"], "IoU", result["native_raw"]["mean_iou"])
    print(json.loads((run / "pilot/summary.json").read_text()))
